# Design exercise D2 · Price suggestions — Part B

Usato suggests a price to every seller who creates a listing. Your job: for
each of 4,000 new listings, predict the price it will sell for, as the
**natural log** of the price in euros.

You have about 15 minutes. The cells below load the data, define the metric,
cross-validate a baseline and write the file. Make **your** design choices in
the cell marked *Your model*, compare them on the folds, then hand in.

**Scoring.** Your predictions are compared with the log of the price each
listing really sold for, which only the grader has. The root mean squared
error gives the points: 0 at the level of one price for every listing (1.56),
10 at the reference model's 0.44 or below, linear in between.

> **One attempt.** Only the **first** file you turn in is scored. If you
> unsubmit and turn in another file, it is not scored: you keep the score of
> the first. Choose your model with the notebook's own validation, check the file with the last
> cell, and only then turn it in.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

BASE = "https://raw.githubusercontent.com/fabsilvestri/AIML-Course/main/design/d02/"
train = pd.read_csv(BASE + "train.csv")       # sold listings, with the sale price
unsold = pd.read_csv(BASE + "unsold.csv")     # expired unsold, with the asking price
new = pd.read_csv(BASE + "test.csv")          # the 4,000 listings to price

assert train.shape == (8000, 10) and new.shape == (4000, 9)
assert unsold.shape == (3000, 10)
assert (train["sale_price_eur"] >= 1).all()
print(f"{len(train):,} sold listings, {len(unsold):,} unsold; "
      f"{len(new):,} new listings to price")
train.head()

The metric is the one the hand-in is scored on: the root mean squared error of
the natural log of the sale price. Cross-validation uses five folds, shuffled
with a fixed seed.

In [ ]:
def rmse(y, pred):
    """Root mean squared error, on whatever scale y and pred are given."""
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(pred)) ** 2)))

X = train.drop(columns=["listing_id", "sale_price_eur"])
y = np.log(train["sale_price_eur"].to_numpy())       # the scale of the hand-in
folds = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_rmse(model):
    """Out-of-fold predictions from cross_val_predict, then the metric."""
    return rmse(y, cross_val_predict(model, X, y, cv=folds))

A baseline, to beat: one price per category, close to what the app suggests
today. Linear regression on the category alone, one-hot encoded, predicts each
category's mean log price.

In [ ]:
numerical = ["age_years", "original_price_eur", "photos", "description_words"]
categorical = ["category", "brand", "condition", "city"]

baseline = make_pipeline(
    make_column_transformer(
        (OneHotEncoder(handle_unknown="ignore"), ["category"])),
    LinearRegression())

print(f"baseline: {cv_rmse(baseline):.3f}")

## Your model

Build your design here, and compare it with the baseline on the folds. Every
choice — which rows, which columns, how each is encoded, which model, which
hyperparameters — is yours, and is judged by `cv_rmse`.

In [ ]:
model = make_pipeline(
    make_column_transformer(
        (OneHotEncoder(handle_unknown="ignore"), ["category"])),
    LinearRegression())          # replace with your design

print(f"your model: {cv_rmse(model):.3f}")

## Hand in

Fit the chosen model on the training rows, predict the 4,000 new listings, and
check the file before you write it.

In [ ]:
model.fit(X, y)
pred = model.predict(new.drop(columns=["listing_id"]))

out = pd.DataFrame({"listing_id": new["listing_id"], "log_price": pred})
assert len(out) == 4000 and out["listing_id"].is_unique
assert set(out["listing_id"]) == set(new["listing_id"])
assert np.isfinite(out["log_price"]).all(), "a prediction is missing or infinite"
assert out["log_price"].between(-1, 11).all(), \
    "log_price must be the natural log of a price in euros, not the price"
out.to_csv("suggestions.csv", index=False)
print("wrote suggestions.csv:", len(out), "rows; median suggestion",
      f"{np.exp(out['log_price'].median()):.0f} euros")

In Colab, download it (the folder icon on the left, or run the next cell),
then in Classroom open the assignment, **Add → File**, choose
`suggestions.csv`, and **Turn in**. The grader marks it within the hour.
**Only this first file counts**: a second one is not scored.

In [ ]:
try:
    from google.colab import files
    files.download("suggestions.csv")
except ImportError:
    print("Not in Colab: suggestions.csv is in the working directory.")